# Week 4, Notebook 1: What Is a Neural Network?

**Building a tiny brain from scratch, one neuron at a time.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A single **neuron** that you code by hand, with no special libraries.
- A picture of the three things every neuron does: **multiply, add, and squash**.
- A neuron that **learns** to predict whether a student in the Caribbean passes an exam.

### The big idea
A neural network is not magic and it is not a real brain. It is a pile of very
simple maths, repeated many times. If you understand **one neuron**, you
understand the whole thing, because a network is just many neurons wired
together.

In this notebook we build that one neuron from the ground up so nothing is
hidden. Next notebook we let a real library (Keras) build a big network for us,
and you will already know what it is doing inside.

### The story: will the student pass?

Meet **Aaliyah**, a student in Kingston, Jamaica. She is about to sit an exam.
We want to guess whether she will **pass** using two clues:

1. How many **hours she studied**.
2. How many **hours she slept** the night before.

A neuron takes these clues (we call them **inputs**), decides how important each
one is (we call these **weights**), and produces a guess between 0 and 1. Close
to **1** means "very likely to pass". Close to **0** means "likely to fail".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Aaliyah's two clues (her "features").
study_hours = 6.0     # she studied 6 hours
sleep_hours = 7.0     # she slept 7 hours

inputs = np.array([study_hours, sleep_hours])
print("Aaliyah's inputs:", inputs)

### Step 1: The neuron multiplies each input by a weight

Every input gets a **weight**: a number that says how much that clue matters.

- A **big positive** weight means "this clue strongly pushes toward PASS".
- A weight near **zero** means "this clue barely matters".
- A **negative** weight means "this clue pushes toward FAIL".

The neuron also has a **bias**, a single extra number that shifts the whole
answer up or down. Think of the bias as the neuron's mood before it sees any
clues: an optimistic neuron (high bias) leans toward pass; a strict one leans
toward fail.

Right now we will just **pick** some weights by hand. Later the neuron will
*learn* its own.

In [ ]:
# Pick some starting weights and a bias by hand.
# We believe studying matters a lot, and sleep matters a little.
weights = np.array([0.5, 0.2])   # weight for study, weight for sleep
bias = -4.0                       # start pessimistic: you must earn a pass

# Multiply each input by its weight, then add them up, then add the bias.
# This one number is called the "weighted sum" or z.
z = np.dot(inputs, weights) + bias
print("Weighted sum z =", z)
print("(that is 6*0.5 + 7*0.2 + (-4.0) =", 6*0.5 + 7*0.2 - 4.0, ")")

### Step 2: The neuron squashes the sum into a 0-to-1 answer

The weighted sum `z` can be any number, huge or tiny. But we want a tidy answer
between 0 and 1 (a probability of passing). So we pass `z` through a **squashing
function**. The most famous one is the **sigmoid**.

The sigmoid takes any number and gently squeezes it into the range 0 to 1:

- Very negative `z` -> answer near **0** (fail).
- `z` of 0 -> answer of exactly **0.5** (a coin flip).
- Very positive `z` -> answer near **1** (pass).

Let us draw it so you can see the S-shape it is named after.

In [ ]:
def sigmoid(x):
    # The squashing function. np.exp is "e to the power of".
    return 1 / (1 + np.exp(-x))

# Draw the sigmoid curve from -10 to 10.
xs = np.linspace(-10, 10, 200)
plt.figure(figsize=(7, 4))
plt.plot(xs, sigmoid(xs), linewidth=2)
plt.axhline(0.5, color="grey", linestyle="--")
plt.axvline(0, color="grey", linestyle="--")
plt.title("The sigmoid: it squashes any number into 0...1")
plt.xlabel("z (the weighted sum)")
plt.ylabel("neuron's answer")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Now squash Aaliyah's weighted sum into a probability.
prediction = sigmoid(z)
print("Chance Aaliyah passes:", round(prediction, 3))

if prediction >= 0.5:
    print("The neuron says: PASS")
else:
    print("The neuron says: FAIL")

### That is the whole neuron

Look how small it was. A neuron does exactly three things:

```
   inputs  --(multiply by weights)-->  add them up (+ bias)  --(squash)-->  answer
```

Let us wrap those three steps into one tidy function so we can reuse it. This
function *is* a neuron.

In [ ]:
def neuron(inputs, weights, bias):
    z = np.dot(inputs, weights) + bias   # multiply and add
    return sigmoid(z)                    # squash

# Try it on a few students from around the Caribbean.
students = {
    "Aaliyah (Jamaica)":        [6, 7],   # studied a lot, slept well
    "Marcus (Barbados)":        [1, 8],   # barely studied
    "Priya (Trinidad)":         [8, 5],   # studied hard, tired
    "Kwame (Guyana)":           [0, 9],   # did not study at all
}

for name, feats in students.items():
    p = neuron(np.array(feats), weights, bias)
    verdict = "PASS" if p >= 0.5 else "fail"
    print(f"{name:22s} study={feats[0]}h sleep={feats[1]}h  ->  {p:.2f}  {verdict}")

### Step 3: But who chooses the weights? The neuron should *learn* them

So far we *guessed* the weights (0.5 and 0.2) and the bias (-4). That worked
okay, but guessing does not scale. The real power of neural networks is that
they **learn their own weights from examples**.

Here is the idea, in plain words:

1. Start with **random** weights (the neuron knows nothing).
2. Show it an example and let it **guess**.
3. Measure how **wrong** the guess was. That number is called the **loss**.
4. Nudge every weight a tiny bit in the direction that would have made the
   loss smaller. This nudging is called **gradient descent**.
5. Repeat thousands of times. Slowly the weights get good.

Let us make a small set of past students (with known results) and teach a
neuron from scratch.

In [ ]:
# Build a small training set: 200 past Caribbean students.
# Rule hidden in the data (the neuron must discover it):
#   a student tends to pass if study*1.0 + sleep*0.5 is high enough.
rng = np.random.default_rng(0)
n = 200
study = rng.uniform(0, 10, n)      # 0 to 10 hours of study
sleep = rng.uniform(3, 10, n)      # 3 to 10 hours of sleep

score = 1.0 * study + 0.5 * sleep + rng.normal(0, 1.0, n)   # hidden rule + luck
passed = (score > 7.5).astype(float)                        # 1 = passed, 0 = failed

X = np.column_stack([study, sleep])    # shape (200, 2): two features per student
y = passed                             # the true answers

print("First 5 students (study, sleep):")
print(X[:5].round(1))
print("Did they pass? (1=yes):", y[:5])
print("Pass rate in the data:", y.mean().round(2))

### A quick clean-up: put the features on the same scale

`study` runs 0-10 and `sleep` runs 3-10. Neural networks learn faster and more
safely when every feature is on a similar, small scale. So we **standardise**:
subtract the average and divide by the spread, turning each column into numbers
centred on 0. This is a habit you will use in every notebook this week.

In [ ]:
means = X.mean(axis=0)
stds = X.std(axis=0)
Xs = (X - means) / stds     # standardised features, centred on 0

print("Before scaling, first row:", X[0].round(2))
print("After scaling,  first row:", Xs[0].round(2))

### Step 4: Train the neuron with gradient descent

Now the heart of it. We will:

- Start with random weights and bias.
- Loop many times (each loop is called an **epoch**):
  - Let the neuron guess for every student.
  - Work out the error (guess minus truth).
  - Slide the weights a little to shrink that error.

The size of each slide is set by the **learning rate**. Too big and the neuron
overshoots; too small and it learns painfully slowly. `0.1` is a sensible start.

You do not need to memorise the gradient formulas below. The one idea to keep is:
*the neuron measures its mistake, then adjusts the weights to make that mistake
smaller, over and over.*

In [ ]:
# Start knowing nothing: random weights, zero bias.
rng = np.random.default_rng(1)
w = rng.normal(0, 0.1, size=2)   # one weight per feature
b = 0.0
learning_rate = 0.1

loss_history = []
for epoch in range(2000):
    # 1) FORWARD PASS: the neuron guesses for all students at once.
    z = Xs @ w + b
    guess = sigmoid(z)

    # 2) LOSS: how wrong are we? (mean squared error, averaged over students)
    error = guess - y
    loss = np.mean(error ** 2)
    loss_history.append(loss)

    # 3) GRADIENTS: which way to nudge w and b to shrink the loss.
    #    (This is calculus done for you; the shapes line up with w and b.)
    d = error * guess * (1 - guess)      # slope of loss through the sigmoid
    grad_w = Xs.T @ d / len(y)
    grad_b = np.mean(d)

    # 4) UPDATE: take a small step downhill.
    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

print("Learned weights:", w.round(3))
print("Learned bias:   ", round(b, 3))
print("Final loss:     ", round(loss_history[-1], 4))

### Watch the neuron get less wrong

The loss should fall as training goes on. That falling curve is the neuron
*learning*. If you ever train a network and the loss does **not** fall, something
is wrong (usually the learning rate or the data).

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(loss_history, linewidth=2)
plt.title("The neuron learning: loss goes down over time")
plt.xlabel("epoch (one full pass over the students)")
plt.ylabel("loss (how wrong, lower is better)")
plt.grid(alpha=0.3)
plt.show()

### How good did it get?

Let us score the trained neuron: for each student, does its guess (rounded to
pass/fail) match the truth? We compare against the **baseline** of always
guessing the more common answer. A model only earns its keep if it beats that
lazy baseline.

In [ ]:
final_guess = sigmoid(Xs @ w + b)
predicted_pass = (final_guess >= 0.5).astype(float)
accuracy = np.mean(predicted_pass == y)

baseline = max(y.mean(), 1 - y.mean())   # always guess the majority class
print(f"Neuron accuracy: {accuracy:.1%}")
print(f"Baseline (always guess majority): {baseline:.1%}")
print("The neuron beat the baseline!" if accuracy > baseline else "No better than guessing.")

### Try it on a brand-new student

Because we scaled the training data, we must scale any new student the same way
(using the same means and spreads we saved). Then the trained neuron gives a
probability of passing.

In [ ]:
def predict_student(study_h, sleep_h):
    raw = np.array([study_h, sleep_h])
    scaled = (raw - means) / stds          # SAME scaling as training
    p = sigmoid(scaled @ w + b)
    return p

for name, feats in [("Shanice (Bahamas)", (7, 8)),
                    ("Diego (Cuba)",      (2, 6)),
                    ("Anaya (Grenada)",   (9, 4))]:
    p = predict_student(*feats)
    print(f"{name:20s} study={feats[0]}h sleep={feats[1]}h  ->  {p:.0%} chance to pass")

### What you learned

- A **neuron** does three tiny things: multiply inputs by weights, add them up
  (with a bias), and squash the result into a 0-to-1 answer.
- The **sigmoid** is the squasher that gives a probability.
- A neuron **learns** by measuring its error (the **loss**) and nudging its
  weights downhill with **gradient descent**, over many **epochs**.
- Always **scale** your features, and always **beat a baseline**.

A neural network is just many of these neurons stacked in **layers**. Wiring them
by hand would be painful, so in the next notebook we hand that job to **Keras**
and build a real **deep** network to predict Caribbean World Cup matches.

### Try it yourself
1. Change `learning_rate` to `0.001`. What happens to the loss curve?
2. Add a third input, `revision_classes_attended`, to the training data and
   retrain. Does accuracy improve?
3. Change the hidden rule in the data (say, make sleep matter more) and see if
   the neuron discovers the new weights.